# 04 · Higher-order interactions (O-information)

Visualises `src/higher_order` output (run `python -m src.higher_order.run`). Gaussian O-information of all region
triplets: Ω > 0 redundancy-dominated, Ω < 0 synergy-dominated.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pickle
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from src.core import paths
from src.core.data import GROUPS, load_data, load_model_results, load_features

sns.set_theme(style="whitegrid", context="notebook")
PALETTE = {"HC": "#2a9d8f", "AP": "#e76f51", "NAP": "#e9c46a"}

In [ ]:
from src.core.measures import triplet_oinfo, copula_normalize
D = paths.DERIVED / "higher_order"
S = pd.read_csv(D / "subject_measures.csv")
G = pd.read_csv(D / "group_tests.csv")
Gc = pd.read_csv(D / "group_tests_compositions.csv")
MV = pd.read_csv(D / "model_vs_empirical.csv")
display(G.round(4))

## Global O-information by group

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, m in zip(axes, ("Omega_mean", "syn_frac", "TC_mean")):
    sns.boxplot(data=S, x="group", y=m, order=GROUPS, palette=PALETTE, ax=ax, showfliers=False)
    sns.stripplot(data=S, x="group", y=m, order=GROUPS, color="k", alpha=.35, size=3, ax=ax)
    r = G.set_index("measure").loc[m]; ax.set_title(f"{m}  (p={r.p_perm:.3f})"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

## Triplet O-information distribution, one subject: empirical vs GEC model

In [ ]:
ts, _, _ = load_data(); res = load_model_results()["results"]
g, k = "NAP", 0
om_cop, _ = triplet_oinfo(np.corrcoef(copula_normalize(ts[g][k])))
om_emp, _ = triplet_oinfo(np.corrcoef(ts[g][k]))
om_mod, _ = triplet_oinfo(res[g][k]["FC_sim"])
fig, ax = plt.subplots(figsize=(9, 4))
bins = np.linspace(-0.3, 0.6, 120)
for v, lab in ((om_cop, "empirical (copula)"), (om_emp, "empirical (Pearson)"), (om_mod, "GEC model")):
    ax.hist(v, bins=bins, histtype="step", lw=1.5, density=True, label=f"{lab}: {np.mean(v < 0):.1%} synergistic")
ax.axvline(0, color="k", lw=.8); ax.set_xlabel("Ω (nats)"); ax.legend()
plt.tight_layout(); plt.show()

## Can the linear model produce synergy?

In [ ]:
display(MV.groupby("group").mean(numeric_only=True).round(4))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sns.scatterplot(data=MV, x="Omega_mean_emp", y="Omega_mean_model", hue="group", palette=PALETTE, ax=axes[0])
lim = [MV[["Omega_mean_emp", "Omega_mean_model"]].min().min(), MV[["Omega_mean_emp", "Omega_mean_model"]].max().max()]
axes[0].plot(lim, lim, "k--", lw=.8)
sns.boxplot(data=MV.melt(id_vars="group", value_vars=["syn_frac_emp", "syn_frac_model"]), x="variable", y="value",
            hue="group", palette=PALETTE, ax=axes[1])
axes[1].set_ylabel("fraction synergistic triplets")
plt.tight_layout(); plt.show()

## Network compositions with the largest group effects

In [ ]:
top = Gc.reindex(Gc.g_NAP_vs_HC.abs().sort_values(ascending=False).index).head(20)
fig, ax = plt.subplots(figsize=(8, 7))
d = top.melt(id_vars="composition", value_vars=["g_AP_vs_HC", "g_NAP_vs_HC"], var_name="contrast", value_name="g")
sns.barplot(data=d, y="composition", x="g", hue="contrast", palette=[PALETTE["AP"], PALETTE["NAP"]], ax=ax)
ax.axvline(0, color="k", lw=.8); ax.set_title("mean Ω per network triplet: Hedges g vs HC (top 20 |g| NAP)")
plt.tight_layout(); plt.show()
display(top.round(4))

## Symptom correlations

In [ ]:
display(pd.read_csv(D / "symptom_correlations.csv").sort_values("p_perm").round(4))